In [1]:
# ===================================================
# MODULE 2.5: OVERFITTING & BIAS-VARIANCE DIAGNOSIS
# ===================================================

import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load dataset and model artifacts
DATA_FILE_PATH = r"C:\Users\hp\Desktop\CG_Final_Project\Dataset\processed\final_numeric_retail_inventory.csv"
MODEL_PATH = "demand_forecast_model.pkl"

if not os.path.exists(DATA_FILE_PATH) or not os.path.exists(MODEL_PATH):
    raise FileNotFoundError("Make sure 'final_numeric_retail_inventory.csv' and 'demand_forecast_model.pkl' are in this directory.")

df = pd.read_csv(DATA_FILE_PATH)
model = joblib.load(MODEL_PATH)

# 2. Align Features and Target to the original configuration
target_column = 'Units Sold'
columns_to_drop = [target_column]
if 'Date' in df.columns:
    columns_to_drop.append('Date')

X = df.drop(columns=columns_to_drop)
y = df[target_column]

# Re-create identical train-test splits using the same seed (random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Generate predictions for both subsets
print("🔮 Generating predictions for Train and Test sets...")
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# 4. Calculate comparative metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# 5. Display comparative results
print("\n" + "="*23 + " OVERFITTING DIAGNOSIS REPORT " + "="*23)
print(f"{'Evaluation Metric':<30} | {'Training Set (Seen)':<20} | {'Testing Set (Unseen)':<20}")
print("-"*76)
print(f"{'Mean Absolute Error (MAE)':<30} | {train_mae:<20.2f} | {test_mae:<20.2f}")
print(f"{'Root Mean Squared Error (RMSE)':<30} | {train_rmse:<20.2f} | {test_rmse:<20.2f}")
print(f"{'R² Score (Accuracy Matrix)':<30} | {train_r2:<20.4f} ({train_r2*100:.2f}%) | {test_r2:<20.4f} ({test_r2*100:.2f}%)")
print("="*76)

# 6. Interpretative Diagnostics Automation
r2_variance = train_r2 - test_r2
print(f"\n📈 Variance Delta (Train R² - Test R²): {r2_variance:.4f}")

if r2_variance > 0.05:
    print("⚠️ DIAGNOSIS: High Variance Detected! The model performs drastically better on training data.")
    print("   This indicates OVERFITTING. Consider shortening tree depth (max_depth) or increasing min_samples_split.")
elif train_r2 < 0.70:
    print("📉 DIAGNOSIS: High Bias Detected! The model performs poorly on both training and validation sets.")
    print("   This indicates UNDERFITTING. Consider engineering more complex non-linear interaction features.")
else:
    print("✨ DIAGNOSIS: Ideal Generalization! Performance metrics are tightly synchronized between sets.")
    print("   No harmful overfitting is occurring. Your model is highly reliable and ready for deployment.")

🔮 Generating predictions for Train and Test sets...

======================= OVERFITTING DIAGNOSIS REPORT =======================
Evaluation Metric              | Training Set (Seen)  | Testing Set (Unseen)
----------------------------------------------------------------------------
Mean Absolute Error (MAE)      | 2.72                 | 7.28                
Root Mean Squared Error (RMSE) | 3.26                 | 8.57                
R² Score (Accuracy Matrix)     | 0.9991               (99.91%) | 0.9938               (99.38%)

📈 Variance Delta (Train R² - Test R²): 0.0053
✨ DIAGNOSIS: Ideal Generalization! Performance metrics are tightly synchronized between sets.
   No harmful overfitting is occurring. Your model is highly reliable and ready for deployment.
